# Simulations

A **simulation** runs one workflow *against another* and lets them **talk to each other**: the
**selected** workflow plays the assistant (A), the **counter** workflow plays the simulated caller
(B), and each turn one workflow's reply is fed to the other as its next message. This is how you
exercise an assistant against a scripted "user" and inspect the resulting conversation.

This notebook walks the full arc:

1. Build a 2-node **workflow A** (the assistant)
2. Build a 2-node **workflow B** (the simulated caller)
3. Register a **simulation** of A against B
4. **Run** it and wait for the conversation to finish
5. **Inspect** the interaction transcript and the results
6. **Clean up** every resource

> The typed config classes come from `interactly.configs` (requires `pip install "interactly[configs]"`).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import asyncio
import json

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## A tiny workflow builder

Both A and B are **2-node conversational workflows** with the same shape:

1. A **conversational first node** that `wait`s for the other side and keeps talking — it has
   `self_loop=True`, so as long as its exit condition is unmet it re-runs on every turn (one
   exchange per turn).
2. A **conditional edge** carrying a **free-form (natural-language) condition**. Each turn the
   engine judges the condition against the conversation so far; while it stays *false* the first
   node self-loops and the two workflows keep exchanging messages. When it becomes *true* the edge
   fires and control moves to…
3. A **terminal closing node** (`self_loop=False`, `wait_for_user_message=False`) that delivers one
   final message and then **ends the workflow** — which stops the simulation.

This is the key difference from a plain `DirectEdgeConfig` (which advances after a single execution
and caps the exchange at two messages): the **self-loop + free-form conditional edge** is what lets
the dialogue run for *as many turns as the task needs* before wrapping up.

> `self_loop=True` requires `wait_for_user_message=True` (a self-looping node that never waits would
> be an infinite LLM loop — the runtime rejects it).

In [ ]:
from interactly.configs import (
    SayLLMNodeConfig,
    PromptConfig,
    ConditionalEdgeConfig,
    ConditionConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    OpenAILLMConfig,
    OPENAIModel,
)
from interactly.types.workflows.workflow import Workflow


def two_turn_workflow(name: str, first: tuple[str, str], second: tuple[str, str], exit_condition: str):
    """Build a 2-node workflow: node1 --conditional(free-form)--> node2, each an LLM 'say' node.

    `first` and `second` are (node_name, prompt) pairs; `exit_condition` is a natural-language
    condition string.

    node1 is the conversational node: it `self_loop`s and waits for the other side, so while
    `exit_condition` is judged false it keeps talking (one exchange per turn). When the condition
    becomes true, the conditional edge advances to node2 — a terminal node that delivers a closing
    message and ends the workflow (`wait_for_user_message=False`, so it does not loop forever).
    """
    llm = OpenAILLMConfig(model=OPENAIModel.GPT_5_4_MINI, max_tokens=120)
    node1 = SayLLMNodeConfig(
        name=first[0],
        is_start=True,
        self_loop=True,              # keep conversing until the conditional edge fires
        wait_for_user_message=True,  # required whenever self_loop=True
        main_response_config=PromptConfig(prompt=first[1]),
        llms_config=llm,
    )
    node2 = SayLLMNodeConfig(
        name=second[0],
        self_loop=False,              # terminal node: run once…
        wait_for_user_message=False,  # …then end the thread (True here would self-loop forever)
        main_response_config=PromptConfig(prompt=second[1]),
        llms_config=llm,
    )
    return WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(name=name),
        node_configs=[node1, node2],
        edge_configs=[
            ConditionalEdgeConfig(
                name=f"{first[0]} -> {second[0]}",
                source_node_logical_id=node1.logical_id,
                destination_node_logical_id=node2.logical_id,
                condition=ConditionConfig(condition_freeform=exit_condition),
            )
        ],
    )

## 1. Workflow A — the assistant (a pizza order bot)

A works through a **five-item order** — size, crust, toppings, a drink, and pickup-vs-delivery —
asking for **one detail per message**. Its conversational node self-loops until the free-form
condition ("all five collected") is met, then it confirms the order with a total and says goodbye.
Because it must gather five things one at a time, the conversation naturally runs several turns.
This is the workflow we want to evaluate — it becomes the simulation's **selected** workflow.

In [ ]:
workflow_a: Workflow = await client.workflows.create_from_config(
    two_turn_workflow(
        "Pizza Order Bot (09_simulations)",
        first=(
            "Take Order",
            "You are a pizza shop order taker on a phone call. Collect the caller's order ONE "
            "detail at a time, in this order: (1) pizza size, (2) crust type, (3) toppings, "
            "(4) a drink, (5) pickup or delivery. Ask for exactly ONE of these per message and "
            "wait for the answer before asking the next. Never ask for two things at once and "
            "do NOT confirm the order until you have all five. Keep each message under 20 words.",
        ),
        second=(
            "Confirm & Close",
            "Read the full order back to the caller — size, crust, toppings, drink, and "
            "pickup/delivery — with a made-up total, then thank them and say goodbye. "
            "Keep it under 40 words.",
        ),
        exit_condition=(
            "The customer has provided ALL FIVE of: pizza size, crust type, toppings, a drink, "
            "and whether it is pickup or delivery. Do NOT trigger until every one of the five is "
            "known — if any is still missing, keep asking."
        ),
    )
)
WF_A_ID = workflow_a.id
print(f"Workflow A (assistant) id={WF_A_ID}  name={workflow_a.name!r}")

## 2. Workflow B — the simulated caller (a hungry customer)

B role-plays a customer with a fixed order in mind who answers **only the single detail the bot
asks for**, one at a time — never volunteering everything at once. This deliberate pacing is what
forces the back-and-forth to run for many turns. B self-loops until the bot confirms the order,
then says goodbye. It becomes the simulation's **counter** workflow — the "user" side of the
conversation.

In [ ]:
workflow_b: Workflow = await client.workflows.create_from_config(
    two_turn_workflow(
        "Hungry Customer (09_simulations)",
        first=(
            "Place Order",
            "You are a customer calling a pizza shop to place ONE order. Your order is: a LARGE, "
            "THIN-CRUST pizza with PEPPERONI AND MUSHROOM, a COKE to drink, for DELIVERY. "
            "Answer the order taker's question with ONLY the single detail they asked about, in "
            "under 12 words. Never volunteer other details and never list the whole order at "
            "once — reveal one thing at a time, exactly as asked.",
        ),
        second=(
            "Say Goodbye",
            "The order taker has confirmed your order and given a total. Thank them and end the "
            "call in under 10 words.",
        ),
        exit_condition=(
            "The order taker has read the full order back with a total and/or clearly confirmed "
            "the order is placed. Do NOT trigger while they are still asking for order details."
        ),
    )
)
WF_B_ID = workflow_b.id
print(f"Workflow B (caller) id={WF_B_ID}  name={workflow_b.name!r}")

## 3. Register the simulation of A against B

A `SimulationConfig`-compatible dict wires the two workflows together:

- `selected_workflow` — the assistant under test (**A**)
- `counter_workflow` — the simulated caller it talks to (**B**)
- `assistant_starts_first=True` — A opens the conversation
- `number_of_simulations` — how many independent runs to spawn
- `max_events` / `timeout_seconds` — hard backstops so a run always ends

At run time the engine alternates turns, feeding each workflow's reply to the other as its next
message — that is the "A against B" interaction. Because A collects five order details one at a
time and B answers one at a time, the dialogue now runs **several back-and-forth turns** before
A's exit condition fires; we give `timeout_seconds` extra headroom to cover the dozen-plus
sequential LLM calls this takes.

In [ ]:
config = {
    "name": "Pizza Order Simulation (notebook)",
    "description": "Run the order bot (A) against a simulated customer (B).",
    "selected_workflow": {"workflow_id": WF_A_ID, "version_number": 0},
    "counter_workflow": {"workflow_id": WF_B_ID, "version_number": 0},
    "assistant_starts_first": True,
    "number_of_simulations": 1,
    "max_events": 200,
    "timeout_seconds": 180,  # headroom for the dozen-plus sequential LLM calls a multi-turn order takes
}

simulation = await client.simulations.create(config=config)
SIM_ID = simulation.id
print(f"Simulation id={SIM_ID}  name={simulation.name!r}")

## 4. Run the simulation and wait for it to finish

`run()` starts a **run group** and returns immediately — the conversation executes asynchronously
on the server. We poll `get_run()` until the group reaches a terminal status.

In [ ]:
import asyncio

group = await client.simulations.run(SIM_ID, runner_name="notebook-runner")
RUN_ID = group.id
print(f"Run group id={RUN_ID}  status={group.status}")

TERMINAL = {"completed", "failed", "cancelled"}
for _ in range(80):  # up to ~4 minutes — a multi-turn order runs longer than a 2-message exchange
    group = await client.simulations.get_run(RUN_ID)
    if str(group.status).lower() in TERMINAL:
        break
    await asyncio.sleep(3)
print(f"Final group status: {group.status}")

## 5. Inspect the interaction and results

**5a. Executions.** A group contains `number_of_simulations` individual executions.
`list_detailed_executions()` returns hydrated records — each links the two underlying workflow
runs (A and B) and reports their statuses.

In [ ]:
import json

detailed = await client.simulations.list_detailed_executions(RUN_ID)
print(f"{len(detailed)} execution(s) in this group\n")
for ex in detailed:
    print(f"  run #{ex.get('run_index')}  status={ex.get('status')}")
    print(f"    A (selected): {ex.get('selected_workflow_name')}  [{ex.get('selected_workflow_status')}]")
    print(f"    B (counter):  {ex.get('counter_workflow_name')}  [{ex.get('counter_workflow_status')}]")

execution = detailed[0]
SELECTED_RUN_ID = execution.get("selected_workflow_run_id")
COUNTER_RUN_ID = execution.get("counter_workflow_run_id")

**5b. The conversation transcript.** Each execution is backed by a workflow **run** whose
`input_output_pairs` capture the turn-by-turn exchange. Reading the selected (A) run replays the
whole dialogue: `user_messages` events are what B said (fed in as input), and `assistant_response`
events are A's replies. The helper below prints them in order.

In [ ]:
def print_transcript(run, *, assistant_label: str, user_label: str):
    """Render a run's input_output_pairs as a readable dialogue."""
    for pair in run.input_output_pairs:
        pd = pair.model_dump() if hasattr(pair, "model_dump") else pair
        for event in (pd.get("run_output") or {}).get("events") or []:
            etype = event.get("type")
            if etype == "user_messages":
                for m in event.get("messages") or []:
                    content = m.get("content") if isinstance(m, dict) else None
                    if content:
                        print(f"  {user_label}: {content}")
            elif etype == "assistant_response" and event.get("content"):
                print(f"  {assistant_label}: {event['content']}")


selected_run = await client.runs.get(SELECTED_RUN_ID)
print("Conversation (from A's run — B's lines arrive as user messages):\n")
print_transcript(selected_run, assistant_label="Order Bot (A)", user_label="Customer (B)")

**5c. Evaluation summary.** `evaluation_summary()` aggregates scores for the group across both
workflows. Without evaluation criteria configured on the workflows the numeric summaries are
empty, but the call shows where per-metric averages would appear.

In [ ]:
summary = await client.simulations.evaluation_summary(SIM_ID, group_id=RUN_ID)
print(json.dumps(summary, indent=2, default=str)[:800])

## 6. Cleanup

Delete the simulation config and both throwaway workflows, then close the client.

In [ ]:
del_result = await client.simulations.delete(SIM_ID)
print(f"Simulation deleted: {del_result.get('message', del_result)}")

await client.workflows.delete(WF_A_ID)
await client.workflows.delete(WF_B_ID)
print("Workflows A and B deleted.")

await client.close()

## See also

- Guide: [`../docs/guides/simulations.md`](../docs/guides/simulations.md) — the full `client.simulations` API
  (schema, list/update, stop, and more)
- [`05_workflow_with_tools.ipynb`](05_workflow_with_tools.ipynb) — building and running a single workflow
- [`18_pagination_and_filtering.ipynb`](18_pagination_and_filtering.ipynb) — paging over runs and executions